# Unsupervised learning: clustering and topic models

This notebook accompanies the 2026 unsupervised learning slides. We first examine k-means on simulated texts, then fit and interpret an LDA model on State of the Union speeches. STM and Wordfish are discussed in the slides but are not implemented here.

## K-means on simulated texts

These texts are generated from two word distributions. The generating groups are known here because we simulated them; real corpora rarely come with a true clustering.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(123)

vocabulary = ["banana", "chocolate"]

# Dirichlet priors for two text types
theta_1 = np.random.dirichlet([1, 3])
theta_2 = np.random.dirichlet([3, 1])


In [ ]:
theta_1

In [ ]:
theta_2

In [ ]:

def generate_text(theta, base_length=150, length_jitter=30):
    """
    text generator using multinomial sampling.
    
    theta: word probability vector
    base_length: mean doc length
    length_jitter: Poisson noise for variation
    """
    # Draw random doc length
    w = np.random.poisson(base_length) + length_jitter

    # Sample word counts
    counts = np.random.multinomial(w, theta)

    # Build shuffled text
    words = []
    for word, count in zip(vocabulary, counts):
        words.extend([word] * count)
    np.random.shuffle(words)
    
    return " ".join(words)

# Generate documents
n = 100
texts_1 = [generate_text(theta_1) for _ in range(n)]
texts_2 = [generate_text(theta_2) for _ in range(n)]

all_texts = texts_1 + texts_2
labels = [1]*n + [2]*n

# Build DTM
vectorizer = CountVectorizer(vocabulary=vocabulary)
dtm = vectorizer.fit_transform(all_texts).toarray()

dtm_df = pd.DataFrame(dtm, columns=vocabulary)
dtm_df["class"] = labels

dtm_df.head()


In [ ]:

X = dtm_df[["banana", "chocolate"]].values

X

In [ ]:


def manual_kmeans(X, k=2, max_iter=10, random_state=42):
    """
    Args:
        X (array): Data matrix (n_samples × n_features)
        k (int): number of clusters
        max_iter (int): maximum K-Means iterations
        random_state (int)

    Returns:
        centroid_history (list of arrays): centroids at each iteration
        label_history (list of arrays): cluster assignments per iteration
        final_labels (list): final class assignment
    """

    np.random.seed(random_state)
    n_samples = X.shape[0]

    # Initialization
    initial_idx = np.random.choice(n_samples, k, replace=False)
    centroids = X[initial_idx]

    centroid_history = [centroids.copy()]
    label_history = []

    # Iteration
    for _ in range(max_iter):

        # Assign each point to closest centroid
        distances = np.linalg.norm(X[:, None] - centroids[None, :], axis=2)
        labels_iter = np.argmin(distances, axis=1)
        label_history.append(labels_iter.copy())

        # Update centroids
        new_centroids = np.array([
            X[labels_iter == j].mean(axis=0) if np.any(labels_iter == j) else centroids[j]
            for j in range(k)
        ])

        centroid_history.append(new_centroids.copy())

        # Stop if converged
        if np.allclose(centroids, new_centroids):
            break

        centroids = new_centroids

    return centroid_history, label_history, labels_iter


In [ ]:
centroid_history, label_history, final_labels = manual_kmeans(
    X,
    k=2,
    max_iter=10,
    random_state=666
)

In [ ]:
len(label_history), len(centroid_history), len(final_labels)

In [ ]:
centroid_history

In [ ]:
plt.figure(figsize=(8, 6))

# Determine dynamic k from final centroid shape
k_dynamic = centroid_history[-1].shape[0]

# Plot final clusters
sns.scatterplot(
    x=X[:, 0], 
    y=X[:, 1], 
    hue=final_labels, 
    palette="tab10",
    s=40
)

# Plot centroid movement arrows
for i in range(1, len(centroid_history)):
    old = centroid_history[i - 1]
    new = centroid_history[i]

    for c in range(k_dynamic):
        plt.arrow(
            old[c, 0], old[c, 1],
            new[c, 0] - old[c, 0],
            new[c, 1] - old[c, 1],
            color="black",
            head_width=3,
            length_includes_head=True
        )

# Plot final centroids
for c in range(k_dynamic):
    plt.scatter(
        centroid_history[-1][c, 0],
        centroid_history[-1][c, 1],
        s=200,
        c="black",
        marker="X"
    )

plt.title(f"K-Means Centroid Reassignment (k = {k_dynamic})")
plt.xlabel("Banana Count")
plt.ylabel("Chocolate Count")
plt.show()

In [ ]:
%matplotlib inline
from matplotlib import rc
rc('animation', html='jshtml')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import seaborn as sns

sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(8, 6))

def update(frame):
    ax.clear()

    labels = label_history[frame]
    centroids = centroid_history[frame]

    # extract the number of clusters from centroids shape
    k_dynamic = centroids.shape[0]

    ax.scatter(
        X[:, 0],
        X[:, 1],
        c=labels,
        cmap="tab10",   # tab10 supports up to 10 clusters
        s=40
    )

    ax.scatter(
        centroids[:, 0],
        centroids[:, 1],
        s=200,
        c="black",
        marker="X"
    )

    # Draw arrows 
    if frame > 0:
        old = centroid_history[frame - 1]
        new = centroid_history[frame]

        for c in range(k_dynamic):
            ax.arrow(
                old[c, 0], old[c, 1],
                new[c, 0] - old[c, 0],
                new[c, 1] - old[c, 1],
                color="black",
                head_width=3,
                length_includes_head=True
            )

    # Axes formatting
    ax.set_xlim(X[:, 0].min() - 10, X[:, 0].max() + 10)
    ax.set_ylim(X[:, 1].min() - 10, X[:, 1].max() + 10)
    ax.set_xlabel("Banana Count")
    ax.set_ylabel("Chocolate Count")
    ax.set_title(f"K-Means Iteration {frame} (k = {k_dynamic})")

ani = FuncAnimation(
    fig,
    update,
    frames=len(label_history),
    interval=1200,
    repeat=False
)

ani


### Changing the data changes the clustering

The third simulated group has longer documents. The model uses raw counts and Euclidean distance, so length can influence clustering alongside word proportions. Compare this representation with normalized frequencies.

In [ ]:
# Add more text data

n = 100
vocabulary = ["banana", "chocolate"]

# Cluster 1: Dirichlet([1,2]), average length ~150
text_type_1 = [
    generate_text(
        theta=np.random.dirichlet([1, 2]),
        base_length=150,
        length_jitter=30
    )
    for _ in range(n)
]

# Cluster 2: Dirichlet([2,1]), average length ~150
text_type_2 = [
    generate_text(
        theta=np.random.dirichlet([2, 1]),
        base_length=150,
        length_jitter=30
    )
    for _ in range(n)
]

# Cluster 3: Dirichlet([50,50]), longer docs (~250 words)
text_type_3 = [
    generate_text(
        theta=np.random.dirichlet([50, 50]),
        base_length=250,
        length_jitter=40
    )
    for _ in range(500)
]

# Combine
all_texts = text_type_1 + text_type_2 + text_type_3
labels = [1]*n + [2]*n + [3]*500


In [ ]:
dtm = vectorizer.fit_transform(all_texts).toarray()
dtm_df = pd.DataFrame(dtm, columns=vocabulary)
dtm_df["class"] = labels

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=dtm_df,
    x="banana", y="chocolate",
    hue="class", palette="Set2", legend="full"
)
plt.title("Three-Cluster Synthetic Text Data (Variable Length)")
plt.xlabel("Banana Count")
plt.ylabel("Chocolate Count")
plt.show()


In [ ]:
X = dtm_df[["banana", "chocolate"]].values
k = 3
max_iter = 10


centroid_history, label_history, final_labels = manual_kmeans(
    X,
    k=k,
    max_iter=10,
    random_state=666
)


In [ ]:
ani = FuncAnimation(
    fig,
    update,
    frames=len(label_history),
    interval=1200,
    repeat=False
)

ani

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def compute_wss(data, max_k=15):
    """
    Compute Within-Cluster Sum of Squares (WSS) for KMeans with k = 1..max_k.
    """
    wss = []
    for k in range(1, max_k + 1):
        model = KMeans(n_clusters=k, n_init=10, random_state=123)
        model.fit(data)
        wss.append(model.inertia_)   # WSS
    return wss

# Compute WSS
max_k = 15
wss_values = compute_wss(dtm, max_k)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(range(1, max_k + 1), wss_values, marker='o')
plt.title("Elbow Method: WSS vs Number of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.axvline(3, color='red', linestyle='--', label='Three simulated groups')
plt.ylabel("Within-Cluster Sum of Squares (WSS)")
plt.xticks(range(1, max_k + 1))
plt.legend()
plt.grid()
plt.show()


# Topic Modeling

## From word counts to topics

We fit LDA to State of the Union speeches from 1901 onward. Documents are mixtures of topics; topics are distributions over words. We use spaCy to tokenize and lemmatize, and Gensim to fit the model. Preprocessing choices can change the topics, so inspect both the words and representative speeches.

**Setup.** Install the Python packages `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`, `spacy`, `gensim`, and `pyLDAvis`, and the spaCy model `en_core_web_md`. For example, run `python -m spacy download en_core_web_md` in a terminal. Run the notebook from this folder, where `sotu.csv` is located.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_md", disable=["ner", "parser"])

In [ ]:
raw_text = "Hello, my name is Peter, I love APPLES!!!!!!!"



In [ ]:
parsed_text = nlp(raw_text)

In [ ]:
for token in parsed_text:
    print(token)

In [ ]:
for token in parsed_text:
    print(token.lemma_)

In [ ]:
for token in parsed_text:
    print(token.is_stop)

In [ ]:

def spacy_preprocess(text):
    """
    - tokenize
    - lowercase
    - remove stopwords
    - remove punctuation
    - lemmatize
    """
    doc = nlp(text)
    tokens = [
        tok.lemma_.lower()
        for tok in doc
        if tok.is_alpha
        and not tok.is_stop
    ]
    return tokens

In [ ]:
# Load the text data
sotu = pd.read_csv('sotu.csv')
# Inspect the data
print(sotu.head())


print(sotu['text'][0])

sotu = sotu[sotu['year'] >= 1901].reset_index(drop=True)

In [ ]:
# Process texts in batches; disable expensive components above.
processed_docs = [[tok.lemma_.lower() for tok in doc if tok.is_alpha and not tok.is_stop]
                  for doc in nlp.pipe(sotu['text'].fillna('').astype(str), batch_size=8)]

In [ ]:
processed_docs[1]

In [ ]:
from gensim.corpora import Dictionary

# Create dictionary (word → id mapping)
dictionary = Dictionary(processed_docs)

dictionary.filter_extremes(
    no_below=5,      # remove words appearing in fewer than 5 documents
    no_above=0.95     # remove words appearing in more than 95% of documents
)

# Bag-of-words corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

In [ ]:
dictionary

In [ ]:
corpus

In [ ]:
from gensim.models.ldamodel import LdaModel

num_topics = 15

lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    passes=10,
    random_state=42
)

In [ ]:
for i, topic in lda.show_topics(num_topics=num_topics, num_words=10, formatted=False):
    print(f"Topic {i}: {[word for word, _ in topic]}")

In [ ]:
lda.print_topics(num_topics=5, num_words=5)

In [ ]:
doc_topics = [lda.get_document_topics(bow) for bow in corpus]

doc_topics

### Interactive topic visualization

The inter-topic distances and word lists provide a starting point for interpretation. Read representative documents too: coherent word lists alone do not establish substantive validity.

In [ ]:

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis


In [ ]:
# Prepare the visualization
vis = gensimvis.prepare(lda, corpus, dictionary)

pyLDAvis.enable_notebook()

vis
# Save the visualization to an HTML file
# pyLDAvis.save_html(vis, 'lda_visualization.html')

### Optional comparison: removing named entities

Named entities can dominate topics. The following cells fit a second model after removing recognized entities. This is a different preprocessing choice, not automatically an improvement. The spaCy model's entity recognition is imperfect. This section can take several minutes.

In [ ]:
nlp = spacy.load("en_core_web_md", disable=["parser"])

def spacy_preprocess_no_ner(text):
    doc = nlp(text)
    tokens = []
    
    for tok in doc:
        # Remove tokens that belong to a named entity
        if tok.ent_type_ != "" and tok.ent_iob_ != "O":
            continue

        if tok.is_stop or tok.is_punct or tok.is_space:
            continue
        
        tokens.append(tok.lemma_)
    
    return tokens

In [ ]:
spacy_preprocess_no_ner("Barack Obama visited New York to give a speech. USA, Great Britain, France")

In [ ]:
processed_docs_no_ner = [
    [tok.lemma_.lower() for tok in doc
     if not tok.ent_type_ and tok.is_alpha and not tok.is_stop]
    for doc in nlp.pipe(sotu['text'].fillna('').astype(str), batch_size=8)
]

In [ ]:
dictionary_no_ner = Dictionary(processed_docs_no_ner)

dictionary_no_ner.filter_extremes(
    no_below=5,      # remove words that appear in <= docs
    no_above=0.95     # remove words that appear in >= 95% of the docs
)

# Bag-of-words corpus
corpus_no_ner = [dictionary_no_ner.doc2bow(doc) for doc in processed_docs_no_ner]


num_topics = 10

lda_no_ner = LdaModel(
    corpus=corpus_no_ner,
    id2word=dictionary_no_ner,
    num_topics=num_topics,
    passes=10,
    random_state=42
)

In [ ]:
vis = gensimvis.prepare(lda_no_ner, corpus_no_ner, dictionary_no_ner)

pyLDAvis.enable_notebook()

vis

### Comparing topic models

Topic number and priors affect the result. Coherence is one diagnostic, not proof that a topic has a valid substantive interpretation. Compare representative texts, stability, and external evidence as well.

In [ ]:
# A small, illustrative comparison. More combinations take substantially longer.
topic_range = [5, 10, 15]
alpha_values = ['symmetric']
eta_values = ['symmetric']
passes_values = [5]
# Gensim uses alpha for document-topic and eta for topic-word priors.
# The slides use beta and alpha, respectively.

In [ ]:
from gensim.models import CoherenceModel

def evaluate_lda_model(corpus, dictionary, texts, num_topics, alpha, eta, passes):
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=num_topics,
        random_state=42,
        chunksize=2000,
        iterations=400,
        alpha=alpha,
        eta=eta,
        passes=passes,
    )
    
    # Compute coherence score
    coher = CoherenceModel(
        model=lda, 
        texts=texts, 
        dictionary=dictionary, 
        coherence='c_v', processes=1
    ).get_coherence()
    
    return coher, lda

In [ ]:
# Optional: fit the candidate models and compare coherence.
# Set to True to run. This can take several minutes.
run_model_comparison = False
results = []
if run_model_comparison:
    for k in topic_range:
        for a in alpha_values:
            for eta in eta_values:
                for p in passes_values:
                    coherence, model = evaluate_lda_model(
                        corpus, dictionary, processed_docs, k, a, eta, p)
                    results.append({'k': k, 'alpha': a, 'eta': eta,
                                    'passes': p, 'coherence': coherence})
    display(pd.DataFrame(results).sort_values('coherence', ascending=False))
else:
    print('Model comparison skipped. Set run_model_comparison = True to run it.')